# 6.22 Gradient accumulation & activation checkpointing

Accumulation trades time for larger effective batches; checkpointing trades extra recompute for lower activation memory.

The optimizer can see a large effective batch even when memory only permits smaller micro-batches. Save a copy to Drive to edit.

In [ ]:

import math
import time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

np.random.seed(42)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def pad_to_64(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 64:
        return X.copy()
    if X.shape[1] > 64:
        return X[:, :64].copy()
    out = np.zeros((X.shape[0], 64), dtype=float)
    out[:, :X.shape[1]] = X
    return out


def split_scale(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te




def softmax_logits(X, W):
    logits = X @ W
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def train_small_mlp_loss(X, y, random_state=0, max_iter=90):
    x_tr, x_te, y_tr, y_te = split_scale(X, y)
    clf = MLPClassifier(hidden_layer_sizes=(16,), activation="relu", solver="adam", alpha=0.001, max_iter=max_iter, random_state=random_state)
    clf.fit(x_tr, y_tr)
    proba = clf.predict_proba(x_te)
    loss = log_loss(y_te, proba, labels=clf.classes_)
    acc = accuracy_score(y_te, clf.predict(x_te))
    return float(loss), float(acc), clf


def train_small_mlp_acc(X, y, random_state=0, max_iter=90):
    loss, acc, clf = train_small_mlp_loss(X, y, random_state=random_state, max_iter=max_iter)
    return acc, loss, clf


def preview_ladder(rungs):
    for rung, (name, X, y) in enumerate(rungs, 1):
        classes = np.unique(y)
        print(f"D{rung}: {name:36s} X={X.shape} classes={len(classes)} sample_y={classes[:5].tolist()}")


def plot_metric_table(rows, metric_name):
    print(f"{'rung':<4} {'dataset':<36} {metric_name:>10}")
    for row in rows:
        print(f"{row['rung']:<4} {row['name']:<36} {row['metric']:10.4f}")


def plot_summary(rows, metric_name, ylabel):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    names = [row["rung"] for row in rows]
    values = [row["metric"] for row in rows]
    axes[0].bar(names, values, color="steelblue")
    axes[0].set_title("Metric by ladder rung")
    axes[0].set_ylabel(ylabel)
    axes[1].plot(names, values, marker="o")
    axes[1].set_title("D1→D5 trend")
    axes[1].set_ylabel(ylabel)
    fig.tight_layout()
    plt.show()


def show_artifacts(rungs, title):
    fig, axes = plt.subplots(1, len(rungs), figsize=(13, 2.6))
    for ax, (name, X, y) in zip(axes, rungs):
        if X.shape[1] == 64:
            ax.imshow(X[0].reshape(8, 8), cmap="gray")
            ax.set_title(name.split()[0])
        else:
            ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="tab10")
            ax.set_title(name.split()[0])
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


## The concept, built once

Accumulation averages micro-batch gradients: $g=\frac{1}{K}\sum_{k=1}^{K}g_k$. The lesson scalar update is $2.0-0.07\cdot1.35=1.905$.

In [ ]:

def softmax_logits(X, W):
    logits = X @ W
    logits = logits - logits.max(axis=1, keepdims=True)
    return np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)


def softmax_gradient(X, y, W, n_classes):
    P = softmax_logits(X, W)
    Y = np.eye(n_classes)[y]
    return X.T @ (P - Y) / len(X)


def accumulated_gradient(X, y, W, K, n_classes):
    pieces = np.array_split(np.arange(len(X)), K)
    grad = np.zeros_like(W)
    for piece in pieces:
        if len(piece) == 0:
            continue
        grad = grad + softmax_gradient(X[piece], y[piece], W, n_classes) * (len(piece) / len(X))
    return grad

X = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
y = np.array([0, 0, 1, 1])
W = np.array([[0.2, -0.1], [0.3, 0.05]])
full = softmax_gradient(X, y, W, 2)
accum = accumulated_gradient(X, y, W, 2, 2)
updated = 2.0 - 0.07 * 1.35
print("full gradient", full)
print("accumulated gradient", accum)
print("updated parameter", updated)
assert np.allclose(full, accum)
assert np.isclose(updated, 1.9055)


A checkpointed block stores boundaries and recomputes interior activations during backward; here we measure the memory budget explicitly.

In [ ]:

def activation_bytes(batch_size, width, layers, checkpoint_every=None):
    if checkpoint_every is None:
        stored = layers
    else:
        stored = math.ceil(layers / checkpoint_every) + 1
    return int(batch_size * width * stored * 4)

print("no checkpoint bytes", activation_bytes(64, 128, 8))
print("checkpoint every 4 bytes", activation_bytes(64, 128, 8, checkpoint_every=4))


## The dataset ladder

The same code is run from a four-point XOR problem through real 8×8 digit images and a noisy shifted D5 variant.

In [ ]:
rungs = clf_digits_ladder()
preview_ladder(rungs)
show_artifacts(rungs, 'Ladder preview')

## Run the SAME method across D1–D5

In [ ]:

def train_accum_softmax(X, y, accum_steps=4, epochs=80, lr=0.4):
    X = pad_to_64(X)
    x_tr, x_te, y_tr, y_te = split_scale(X, y)
    classes = np.unique(y_tr)
    mapper = {label: i for i, label in enumerate(classes)}
    yt = np.array([mapper[v] for v in y_tr])
    rng = np.random.default_rng(10)
    W = rng.normal(0.0, 0.02, size=(x_tr.shape[1], len(classes)))
    for epoch in range(epochs):
        grad = accumulated_gradient(x_tr, yt, W, accum_steps, len(classes))
        W = W - lr * grad
    P = softmax_logits(x_te, W)
    loss = log_loss(y_te, P, labels=classes)
    preds = classes[np.argmax(P, axis=1)]
    return float(loss), float(accuracy_score(y_te, preds))

rows = []
for idx, (name, X, y) in enumerate(rungs, 1):
    loss, acc = train_accum_softmax(X, y, accum_steps=4)
    rows.append({"rung": f"D{idx}", "name": name, "metric": loss, "accuracy": acc})
plot_metric_table(rows, "loss")


## Results visualization

In [ ]:
show_artifacts(rungs, 'Accumulation artifacts')
plot_summary(rows, 'loss', 'held-out loss')

## Pitfall on D5: forgetting shape and memory

In [ ]:

name, X, y = rungs[-1]
full_bytes = activation_bytes(len(X), 128, 10)
micro_bytes = activation_bytes(64, 128, 10, checkpoint_every=5)
budget = 3500000
print("D5 full-batch activation bytes", full_bytes)
print("micro-batch checkpointed bytes", micro_bytes)
print("budget", budget)
assert full_bytes > budget
assert micro_bytes < budget


## Evaluate it + Practice

- Metric: held-out loss; compare it with a majority-class or plain logistic baseline.
- Sanity check: D1 should be hand-checkable before trusting D5.
- Ablation: turn off the key idea and the reported metric should get worse or the pitfall should reappear.
- Failure signals: unstable losses, collapsed routing, inflated gradient error, or a D5 result that ignores label noise.

Practice 1: change one hyperparameter and rerun the D1 assert before touching D5.

Practice 2: add a baseline row to the ladder table and explain the largest gap.

Practice 3: make the D5 pitfall more severe, then show the same fix still helps.